## Loading and Testing CUDA

In [1]:
import os

# Check CUDA version using nvidia-smi
!nvidia-smi

# Or for a more specific CUDA toolkit version if available
!nvcc --version

Fri Apr 10 04:31:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:1E.0 Off |                    0 |
| N/A   34C    P0             33W /   70W |   11061MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Paddle Version Check

In [2]:
import paddle
import paddleocr # Import the module directly

print("PaddlePaddle version:", paddle.__version__)
print("PaddleOCR version:", paddleocr.__version__)

/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


PaddlePaddle version: 3.3.0
PaddleOCR version: 3.4.0


## Paddle Check

In [32]:
# After runtime restarts, run this cell.
import os
import paddle
from paddleocr import PaddleOCRVL, PaddleOCR
from PIL import Image, ImageOps
import json
import pathlib

print("Paddle version:", paddle.__version__)

# Set device based on GPU availability

if paddle.is_compiled_with_cuda():
    paddle.set_device('gpu')
    print("Paddle device set to GPU.")
    # Run a simple check to verify GPU functionality
    try:
        paddle.utils.run_check()
        print("PaddlePaddle GPU check passed.")
    except Exception as e:
        print(f"PaddlePaddle GPU check failed: {e}")
        print("Falling back to CPU.")
        paddle.set_device('cpu')
else:
    print("No GPU found, falling back to CPU. Please ensure you have selected a GPU runtime (Runtime -> Change runtime type -> Hardware accelerator -> GPU).")
    paddle.set_device('cpu')

Paddle version: 3.3.0
Paddle device set to GPU.
Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
PaddlePaddle GPU check passed.


/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/paddle/pir/math_op_patch.py:241: UserWarning: Tensor do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


## Setup local image location - and Check


In [6]:
# Create workspace
WORKDIR = "./"
pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)
print("Workspace:", WORKDIR)

Workspace: ./


In [28]:
# Using a local demo image for PaddleOCR-VL
local_demo = os.path.join(WORKDIR, "demo3.jpeg")

# Check if the demo image exists locally
if os.path.exists(local_demo):
  print("Demo image found locally.")
else:
  # If the image is not found locally, raise an error as requested
  raise FileNotFoundError(f"Local demo image not found at {local_demo}. Please ensure the image is present.")

# If you want to upload your own image via Colab UI, run this:
# from google.colab import files
# uploaded = files.upload()
# for fn in uploaded: print("Uploaded:", fn)

Demo image found locally.


## Downloading model weights

In [41]:
# Run the PaddleOCRVL pipeline on the remote demo image (will download model weights on first run)
print("Instantiating PaddleOCRVL pipeline (this may download model weights; CPU inference will be slow)...")

# pipeline = PaddleOCRVL(pipeline_version="v1")   # recommended API

# Configure pipeline to use vLLM server
pipeline = PaddleOCRVL(
    vl_rec_backend="vllm-server",
    vl_rec_server_url="http://localhost:8000/v1",
    vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL"  # or "PaddleOCR-VL-0.9B" if you served with that name
)

Creating model: ('PP-DocLayoutV3', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/teamspace/studios/this_studio/.paddlex/official_models/PP-DocLayoutV3`.


Instantiating PaddleOCRVL pipeline (this may download model weights; CPU inference will be slow)...


Creating model: ('PaddleOCR-VL-1.5-0.9B', None)


## Running PaddleOCR-VLM model - on local demo.png

In [46]:

print("Running prediction on demo URL...")
results = pipeline.predict(local_demo)

# results is typically a list of result objects; print summary
print("Number of results returned:", len(results))
for i, r in enumerate(results):
    print(f"--- Result {i} ---")
    # r is a structured object; try to print its string repr and save to files if methods exist
    try:
        print(r)
    except Exception:
        print("Could not print result object directly; converting to dict if possible.")
    # Save JSON/markdown if supported
    try:
        json_path = os.path.join(WORKDIR, f"paddleocr_vl_output1C_{i}.json")
        md_path = os.path.join(WORKDIR, f"paddleocr_vl_output1C_{i}.md")
        r.save_to_json(save_path=json_path)
        r.save_to_markdown(save_path=md_path)
        print("Saved:", json_path, md_path)
    except Exception as e:
        print("Save methods not available or failed:", e)


Running prediction on demo URL...


[2026-04-09 15:29:50,451] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,456] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,457] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,460] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,465] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,466] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,472] [    INFO] _client.py:1740 - HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"
[2026-04-09 15:29:50,481] [    INFO] _client.py:1740 - HTTP Re

Number of results returned: 1
--- Result 0 ---
{'input_path': './demo3.jpeg', 'page_index': None, 'page_count': None, 'width': 660, 'height': 880, 'doc_preprocessor_res': {'output_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], dtype=uint8)}, 'layout_det_res': {'input_path': None, 'page_index': None, 'input_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], dtype=uint8), 'boxes': [{'cls_id': 14, 'label': 'image', 'score': 0.6972392201423645, 'coordinate': [65, 147, 144, 210], 'order': None, 'polygon_points': array([[ 65., 147.],
       ...,
       [ 65., 210.]], dtype=float32)}, {'cls_id': 17, 'label': 'paragraph_title', 'score': 0.4878201484680176, 'coordinate': [152, 150, 303, 207], 'order': 1, 'polygon_points': array([[152., 150.],
       ...,
       [152., 207.]], dtype=float32)}, {'cls

## LOCAL LLM CHAT - ERRORRR

In [ ]:
# Notebook cell: interactive chat with Qwen2.5-1.5B vLLM server
import requests, json, sys

BASE = "http://localhost:8001/v1"
MODEL = "Qwen2.5-1.5B"  # must match --served-model-name

def chat_once(prompt):
    payload = {
      "model": MODEL,
      "messages": [{"role":"user","content": prompt}],
      "temperature": 0.0
    }
    r = requests.post(f"{BASE}/chat/completions", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print("Chat with Qwen2.5-1.5B (type 'exit' to quit)")
history = []
while True:
    user = input("You: ")
    if user.strip().lower() in ("exit","quit"):
        break
    out = chat_once(user)
    print("LLM:", out)


## Testing OLLAMA API Chat

In [ ]:
import requests
import json

# Your Cloudflare Tunnel URL
CLOUDFLARE_URL = "https://post-coding-cruise-tuesday.trycloudflare.com"
MODEL = "llama3.1"  # Ensure this model is pulled on the host machine

def continuous_chat_via_generate():
    context = None  # This stores the "memory" of the chat

    print(f"--- Continuous Chat (via Generate Endpoint) ---")
    print("Type 'quit' to stop.\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['quit', 'exit']:
            break

        payload = {
            "model": MODEL,
            "prompt": user_input,
            "stream": False,
            "context": context  # Send the memory back to Ollama
        }

        try:
            response = requests.post(f"{CLOUDFLARE_URL}/api/generate", json=payload)
            response.raise_for_status()

            data = response.json()

            # Print the AI response
            print(f"AI: {data['response']}\n")

            # Update the context (this is an array of numbers that represents the conversation)
            context = data.get('context')

        except Exception as e:
            print(f"\nError: {e}")
            break

if __name__ == "__main__":
    continuous_chat_via_generate()

## VLM Output Extraction for LLM


In [21]:
import json
import os

# Ensure json_path is defined, it should be from previous cells
json_path = os.path.join(WORKDIR, "paddleocr_vl_output1C_0.json")

# Read the .json file
with open(json_path, 'r') as f:
    ocr_vl_data = json.load(f)

print(f"Successfully loaded JSON from: {json_path}")

Successfully loaded JSON from: ./paddleocr_vl_output1C_0.json


In [22]:
def extract_and_combine_content(data):
    combined_content = []
    if isinstance(data, list) and data:
        # Assuming the structure has a parsing_res_list within the first item
        if 'parsing_res_list' in data[0] and isinstance(data[0]['parsing_res_list'], list):
            for item in data[0]['parsing_res_list']:
                # Access 'content' as an attribute of the PaddleOCRVLBlock object
                if hasattr(item, 'content') and item.content is not None:
                    combined_content.append(item.content)
    return '\n'.join(combined_content)

# Extract and combine the 'content' from the loaded JSON data
# Using 'results' directly which holds the full parsed output from the PaddleOCRVL pipeline.
extracted_block_contents = extract_and_combine_content(results)

print("\n--- Extracted Block Contents ---")
print(extracted_block_contents)


--- Extracted Block Contents ---
invoice

FROM
<table><tr><td>Fku..</td><td>INVOICE #</td><td>US-001</td></tr><tr><td>East Repair Inc.</td><td>INVOICE DATE</td><td>11/02/2019</td></tr><tr><td>1912 Harvest Lane</td><td>P.O.#</td><td>2312/2019</td></tr><tr><td>New York, NY 12210</td><td>DUE DATE</td><td>26/02/2019</td></tr></table>
BILL TO
John Smith
2 Court Square
New York, NY 12210
SHIP TO
John Smith
3787 Pineview Drive
Cambridge, MA 12210
<table><tr><td>QTY</td><td>DESCRIPTION</td><td>UNIT</td><td>PRICE</td><td>AMOUNT</td></tr><tr><td>1</td><td>Front and rear brake cables</td><td>100.00</td><td>100.00</td><td></td></tr><tr><td>2</td><td>New set of pedal arms</td><td>15.00</td><td>30.00</td><td></td></tr><tr><td>3</td><td>Labor 3hrs</td><td>5.00</td><td>15.00</td><td></td></tr><tr><td></td><td></td><td>Subtotal</td><td>145.00</td><td></td></tr><tr><td></td><td></td><td>Sales Tax 6.25%</td><td>9.06</td><td></td></tr><tr><td colspan="3">TOTAL</td><td>$154.06</td><td></td></tr></table>
J

## JSON AI

In [23]:
import json
from openai import OpenAI

# 1. Initialize the OpenAI client pointing to your local vLLM server
client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="EMPTY" # vLLM doesn't need a real key
)

MODEL_NAME = "Qwen2.5-1.5B"

# 2. Define the SYSTEM prompt (This is the part that gets Prefix Cached!)
system_prompt = """You are a precise data extraction assistant. 
Extract the information from the user's text and return it strictly as a JSON object. 
If a field is not found, use null. 
Convert currency values (e.g., $100.00) to float numbers (e.g., 100.00). 
Dates should be in YYYY-MM-DD format.

You must output a valid JSON object using exactly this schema:
{
  "Person_name": "string",
  "Company_name": "string",
  "address": "string",
  "contact": "string",
  "invoice_number": "string",
  "invoice_date": "YYYY-MM-DD",
  "due_date": "YYYY-MM-DD",
  "subtotal": "float",
  "tax": "float",
  "total": "float"
}"""

try:
    print(f"\n--- Calling local vLLM ({MODEL_NAME}) ---")
    
    # 3. Make the API call
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Text to process:\n{extracted_block_contents}"}
        ],
        temperature=0.1, # Low temperature for strict, deterministic extraction
        response_format={"type": "json_object"} # Forces the model to output valid JSON
    )

    # 4. Extract the response text
    raw_output = response.choices[0].message.content
    print("\n--- vLLM API Raw Response ---")
    print(raw_output)

    # 5. Parse it to ensure it is valid Python dictionary/JSON
    parsed_json = json.loads(raw_output)
    print("\n--- Parsed Python Dictionary ---")
    print(json.dumps(parsed_json, indent=2))

except json.JSONDecodeError:
    print("\nError: The model did not return valid JSON.")
    print(f"Raw output was: {raw_output}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


--- Calling local vLLM (Qwen2.5-1.5B) ---


[2026-04-09 15:15:03,319] [    INFO] _client.py:1025 - HTTP Request: POST http://localhost:8001/v1/chat/completions "HTTP/1.1 200 OK"



--- vLLM API Raw Response ---
{
  "Person_name": "John Smith",
  "Company_name": "East Repair Inc.",
  "address": "2 Court Square, New York, NY 12210",
  "contact": "John Smith",
  "invoice_number": "US-001",
  "invoice_date": "2019-11-02",
  "due_date": "2019-02-26",
  "subtotal": 145.00,
  "tax": 9.06,
  "total": 154.06
}

--- Parsed Python Dictionary ---
{
  "Person_name": "John Smith",
  "Company_name": "East Repair Inc.",
  "address": "2 Court Square, New York, NY 12210",
  "contact": "John Smith",
  "invoice_number": "US-001",
  "invoice_date": "2019-11-02",
  "due_date": "2019-02-26",
  "subtotal": 145.0,
  "tax": 9.06,
  "total": 154.06
}


In [ ]:
import requests
import json

# Ensure CLOUDFLARE_URL and MODEL are defined from previous cells
CLOUDFLARE_URL = "https://newsletters-para-represent-rats.trycloudflare.com"
MODEL = "llama3.1"

# Define the custom prompt with the desired JSON format
custom_prompt = f"""Extract the following information from the text below and return it as a JSON object. If a field is not found, use null. Convert currency values (e.g., $100.00) to float numbers (e.g., 100.00). Ensure the output is only the JSON object, with no additional text or formatting. Dates should be in YYYY-MM-DD format.

Desired JSON format example:
```json
{{
  "Person_name": "string",
  "Company_name": "string",
  "address": "string",
  "contact": "string",
  "invoice_number": "string",
  "invoice_date": "DD-MM-YYYY",
  "due_date": "DD-MM-YYYY",
  "subtotal": "float",
  "tax": "float",
  "total": "float"
}}
```

Text to process:
{extracted_block_contents}

Please provide the JSON object directly, with no surrounding text or markdown. Example: {{ "name": "..." }}"""

payload = {
    "model": MODEL,
    "prompt": custom_prompt,
    "stream": False
}

try:
    print("\n--- Calling Ollama API with custom prompt ---")
    response = requests.post(f"{CLOUDFLARE_URL}/api/generate", json=payload)
    response.raise_for_status() # Raise an exception for HTTP errors

    data = response.json()

    # Print only the AI response, which should be the JSON object
    print("\n--- Ollama API Response (JSON Output) ---")
    print(data['response'])

except requests.exceptions.RequestException as e:
    print(f"\nError during API call: {e}")
except json.JSONDecodeError:
    print("\nError: Could not decode JSON response from Ollama API.")
    print(f"Raw response text: {response.text}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


--- Calling Ollama API with custom prompt ---

--- Ollama API Response (JSON Output) ---
{
  "Person_name": "John Smith",
  "Company_name": "East Repair Inc.",
  "address": "1912 Harvest Lane, New York, NY 12210",
  "contact": null,
  "invoice_number": "US-001",
  "invoice_date": "02-11-2019",
  "due_date": "02-26-2019",
  "subtotal": 145.00,
  "tax": 9.06,
  "total": null
}


In [ ]:

# --- Calling Ollama API with custom prompt ---

# --- Ollama API Response (JSON Output) ---
# {
#     "name": "East Repair Inc.",
#     "address": "1912 Harvest Lane New York, NY 12210",
#     "contact": null,
#     "invoice_number": "US-001",
#     "invoice_date": "2019-11-02",
#     "subtotal": 145.00,
#     "tax": 9.06,
#     "total": null
# }

In [45]:
import requests
import json

# Ensure CLOUDFLARE_URL and MODEL are defined from previous cells
CLOUDFLARE_URL = "https://newsletters-para-represent-rats.trycloudflare.com"
MODEL = "llama3.1"

# Define the custom prompt with the desired JSON format
custom_prompt = f"""Extract the following information from the text below and return it as a JSON object. If a field is not found, use null. Convert currency values (e.g., $100.00) to float numbers (e.g., 100.00). Ensure the output is only the JSON object, with no additional text or formatting. Dates should be in YYYY-MM-DD format.

Desired JSON format example:
```json
{{
  "Person_name": "string",
  "Company_name": "string",
  "address": "string",
  "contact": "string",
  "invoice_number": "string",
  "invoice_date": "DD-MM-YYYY",
  "due_date": "DD-MM-YYYY",
  "subtotal": "float",
  "tax": "float",
  "total": "float"
}}
```

Text to process:
{extracted_block_contents}

Please provide the JSON object directly, with no surrounding text or markdown. Example: {{ "name": "..." }}"""

payload = {
    "model": MODEL,
    "prompt": custom_prompt,
    "stream": False
}

try:
    print("\n--- Calling Ollama API with custom prompt ---")
    response = requests.post(f"{CLOUDFLARE_URL}/api/generate", json=payload)
    response.raise_for_status() # Raise an exception for HTTP errors

    data = response.json()

    # Print only the AI response, which should be the JSON object
    print("\n--- Ollama API Response (JSON Output) ---")
    print(data['response'])

except requests.exceptions.RequestException as e:
    print(f"\nError during API call: {e}")
except json.JSONDecodeError:
    print("\nError: Could not decode JSON response from Ollama API.")
    print(f"Raw response text: {response.text}")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


--- Calling Ollama API with custom prompt ---

--- Ollama API Response (JSON Output) ---
{
  "Person_name": "John Smith",
  "Company_name": "East Repair Inc.",
  "address": "2 Court Square New York, NY 12210",
  "contact": null,
  "invoice_number": "US-001",
  "invoice_date": "2019-11-02",
  "due_date": "2019-02-26",
  "subtotal": 145.00,
  "tax": 9.06,
  "total": null
}
